# Day 13: Benchmark LLM Parameter Variance

Welcome to Day 13! Today we dive deep into the knobs and dials that control LLM output behavior: **Temperature**, **Top-P (Nucleus Sampling)**, and **Frequency Penalty**.

## Core Theory (Just-in-Time)

**Why alter these parameters?**
As an AI Engineer, you don't just want text; you want the *right kind* of text. A code generation task requires highly deterministic output, while a creative brainstorming agent requires diverse and surprising ideas.

1. **Temperature:** Controls the "randomness" of the model's predictions. 
   - Technically, it scales the logits before the softmax function is applied.
   - **Low (e.g., 0.0 - 0.3):** Makes the model "greedy", consistently picking the highest-probability token. Best for factual Q&A or code.
   - **High (e.g., 0.7 - 1.5):** Flattens the probability distribution, allowing lower-probability tokens to be selected. Increases creativity but also hallucinations.
2. **Top-P (Nucleus Sampling):** An alternative to temperature. Instead of altering the probabilities, it restricts the sampling pool to the smallest set of tokens whose cumulative probability exceeds `p`.
   - **p=0.9:** The model only considers the tokens making up the top 90% of the probability mass. This dynamically cuts off the "long tail" of highly improbable tokens.
3. **Frequency Penalty:** Penalizes new tokens based on their existing frequency in the generated text so far.
   - Useful for preventing the model from looping or repeating the same phrases. Positive values decrease the likelihood of repetition.

### AI Security & Production Implications
1. **Prompt Injection Risk:** When testing configurations, ensure that untrusted user input is strictly sanitized. Varying temperature doesn't prevent prompt injection; in fact, a high temperature might make the model more susceptible to following creative jailbreaks.
2. **PII Leakage:** If you use real user data to benchmark parameters, ensure PII is scrubbed before invoking the LLM.
3. **Fallback Mechanisms:** In a production environment, API calls can fail due to rate limits or timeouts. Always use `try/except` blocks to handle failures gracefully, ensuring the system can fall back to a default response or retry safely.


## Code Implementation

### 1. Basic: Isolating the Core Concept
A minimal script showing how to vary `temperature`, `top_p`, and `frequency_penalty` across iterations using a simple `for` loop.

In [1]:
import os
from langchain_openai import ChatOpenAI
from langchain_core.messages import HumanMessage

def run_basic_benchmark(prompt: str):
    if not os.environ.get("OPENAI_API_KEY"):
        print("OPENAI_API_KEY missing. Skipping Basic execution.")
        return
        
    configs = [
        {"temperature": 0.0, "top_p": 1.0, "frequency_penalty": 0.0},
        {"temperature": 1.0, "top_p": 1.0, "frequency_penalty": 0.0},
        {"temperature": 1.5, "top_p": 1.0, "frequency_penalty": 1.5}
    ]
    
    for cfg in configs:
        print(f"\n--- Config: {cfg} ---")
        try:
            llm = ChatOpenAI(
                model="gpt-3.5-turbo",
                temperature=cfg["temperature"],
                model_kwargs={
                    "top_p": cfg["top_p"],
                    "frequency_penalty": cfg["frequency_penalty"]
                }
            )
            response = llm.invoke([HumanMessage(content=prompt)])
            print(response.content)
        except Exception as e:
            print(f"API Call Failed: {e}")

run_basic_benchmark("Describe a futuristic city in one sentence.")



--- Config: {'temperature': 0.0, 'top_p': 1.0, 'frequency_penalty': 0.0} ---


/app/.venv/lib/python3.12/site-packages/IPython/core/interactiveshell.py:3775: UserWarning: Parameters {'frequency_penalty', 'top_p'} should be specified explicitly. Instead they were passed in as part of `model_kwargs` parameter.
  exec(code_obj, self.user_global_ns, self.user_ns)


API Call Failed: Error code: 401 - {'error': {'message': 'Incorrect API key provided: sk-dummy. You can find your API key at https://platform.openai.com/account/api-keys.', 'type': 'invalid_request_error', 'code': 'invalid_api_key', 'param': None}, 'status': 401}

--- Config: {'temperature': 1.0, 'top_p': 1.0, 'frequency_penalty': 0.0} ---
API Call Failed: Error code: 401 - {'error': {'message': 'Incorrect API key provided: sk-dummy. You can find your API key at https://platform.openai.com/account/api-keys.', 'type': 'invalid_request_error', 'code': 'invalid_api_key', 'param': None}, 'status': 401}

--- Config: {'temperature': 1.5, 'top_p': 1.0, 'frequency_penalty': 1.5} ---
API Call Failed: Error code: 401 - {'error': {'message': 'Incorrect API key provided: sk-dummy. You can find your API key at https://platform.openai.com/account/api-keys.', 'type': 'invalid_request_error', 'code': 'invalid_api_key', 'param': None}, 'status': 401}


/app/.venv/lib/python3.12/site-packages/IPython/core/interactiveshell.py:3775: UserWarning: Parameters {'frequency_penalty', 'top_p'} should be specified explicitly. Instead they were passed in as part of `model_kwargs` parameter.
  exec(code_obj, self.user_global_ns, self.user_ns)
/app/.venv/lib/python3.12/site-packages/IPython/core/interactiveshell.py:3775: UserWarning: Parameters {'frequency_penalty', 'top_p'} should be specified explicitly. Instead they were passed in as part of `model_kwargs` parameter.
  exec(code_obj, self.user_global_ns, self.user_ns)


### 2. Medium: Clean OOP and State Management
Using Pydantic models to strictly define configuration and state, demonstrating how objects interact cleanly.

In [2]:
import os
from typing import List
from pydantic import BaseModel
from langchain_openai import ChatOpenAI
from langchain_core.messages import HumanMessage

class ConfigModel(BaseModel):
    temperature: float
    top_p: float = 1.0
    frequency_penalty: float = 0.0

class BenchmarkResult(BaseModel):
    config: ConfigModel
    response_text: str
    error: str = ""

class LLMBenchmarker:
    def __init__(self, model_name: str = "gpt-3.5-turbo"):
        self.model_name = model_name
        
    def run_benchmark(self, prompt: str, configs: List[ConfigModel]) -> List[BenchmarkResult]:
        results = []
        for config in configs:
            try:
                llm = ChatOpenAI(
                    model=self.model_name,
                    temperature=config.temperature,
                    model_kwargs={
                        "top_p": config.top_p,
                        "frequency_penalty": config.frequency_penalty
                    }
                )
                response = llm.invoke([HumanMessage(content=prompt)])
                results.append(BenchmarkResult(config=config, response_text=str(response.content)))
            except Exception as e:
                results.append(BenchmarkResult(config=config, response_text="", error=str(e)))
        return results

if __name__ == "__main__":
    if not os.environ.get("OPENAI_API_KEY"):
        print("OPENAI_API_KEY missing. Skipping Medium execution.")
    else:
        configs = [
            ConfigModel(temperature=0.7, top_p=1.0),
            ConfigModel(temperature=1.2, frequency_penalty=1.0)
        ]
        benchmarker = LLMBenchmarker()
        results = benchmarker.run_benchmark("Describe a futuristic city in one sentence.", configs)
        for res in results:
            print(f"Config: {res.config} -> Result: {res.response_text or res.error}")


/app/.venv/lib/python3.12/site-packages/IPython/core/interactiveshell.py:3775: UserWarning: Parameters {'frequency_penalty', 'top_p'} should be specified explicitly. Instead they were passed in as part of `model_kwargs` parameter.
  exec(code_obj, self.user_global_ns, self.user_ns)


Config: temperature=0.7 top_p=1.0 frequency_penalty=0.0 -> Result: Error code: 401 - {'error': {'message': 'Incorrect API key provided: sk-dummy. You can find your API key at https://platform.openai.com/account/api-keys.', 'type': 'invalid_request_error', 'code': 'invalid_api_key', 'param': None}, 'status': 401}
Config: temperature=1.2 top_p=1.0 frequency_penalty=1.0 -> Result: Error code: 401 - {'error': {'message': 'Incorrect API key provided: sk-dummy. You can find your API key at https://platform.openai.com/account/api-keys.', 'type': 'invalid_request_error', 'code': 'invalid_api_key', 'param': None}, 'status': 401}


/app/.venv/lib/python3.12/site-packages/IPython/core/interactiveshell.py:3775: UserWarning: Parameters {'frequency_penalty', 'top_p'} should be specified explicitly. Instead they were passed in as part of `model_kwargs` parameter.
  exec(code_obj, self.user_global_ns, self.user_ns)


### 3. Advanced: Production-Grade Implementation
Features explicit type hinting, logging, full Pydantic validation, exception handling with fallbacks, and exact imports to prepare for rigorous technical interviews.

In [3]:
import os
import logging
from typing import List
from pydantic import BaseModel, Field
from langchain_openai import ChatOpenAI
from langchain_core.messages import HumanMessage

logging.basicConfig(level=logging.INFO, format='%(asctime)s - %(levelname)s - %(message)s')
logger = logging.getLogger(__name__)

class ProdConfigModel(BaseModel):
    """Strict validation for LLM hyperparameters."""
    temperature: float = Field(default=0.7, ge=0.0, le=2.0)
    top_p: float = Field(default=1.0, ge=0.0, le=1.0)
    frequency_penalty: float = Field(default=0.0, ge=-2.0, le=2.0)

class ProdBenchmarkResult(BaseModel):
    """Represents a single iteration result."""
    iteration: int
    config: ProdConfigModel
    response: str
    success: bool

class ProductionLLMBenchmarker:
    """
    A production-ready class to benchmark LLM outputs with varying configurations.
    Includes AI Security best practices: exception handling and safe defaults.
    """
    def __init__(self, model_name: str = "gpt-3.5-turbo"):
        self.model_name = model_name

    def run_benchmark_suite(self, prompt: str, iterations: int = 5) -> List[ProdBenchmarkResult]:
        """
        Executes the prompt across dynamically generated configurations.
        
        Args:
            prompt: The text prompt to test.
            iterations: Number of configurations to test.
            
        Returns:
            A list of ProdBenchmarkResult objects.
        """
        results: List[ProdBenchmarkResult] = []
        
        # Generating a varied set of configs safely
        configs_to_test = [
            ProdConfigModel(temperature=0.0, top_p=1.0, frequency_penalty=0.0), # Greedy
            ProdConfigModel(temperature=0.5, top_p=1.0, frequency_penalty=0.0), # Balanced
            ProdConfigModel(temperature=1.0, top_p=0.5, frequency_penalty=0.0), # Nucleus constraint
            ProdConfigModel(temperature=1.5, top_p=1.0, frequency_penalty=1.0), # Creative but penalized repeats
            ProdConfigModel(temperature=2.0, top_p=1.0, frequency_penalty=0.0)  # Max Randomness
        ]
        
        for i in range(min(iterations, len(configs_to_test))):
            cfg = configs_to_test[i]
            logger.info(f"Running Iteration {i+1} with {cfg}")
            
            try:
                llm = ChatOpenAI(
                    model=self.model_name,
                    temperature=cfg.temperature,
                    model_kwargs={
                        "top_p": cfg.top_p,
                        "frequency_penalty": cfg.frequency_penalty
                    },
                    request_timeout=10.0 # Strict timeout for production
                )
                
                response = llm.invoke([HumanMessage(content=prompt)])
                result_text = str(response.content)
                
                results.append(ProdBenchmarkResult(
                    iteration=i+1,
                    config=cfg,
                    response=result_text,
                    success=True
                ))
                
            except Exception as e:
                logger.error(f"Iteration {i+1} failed: {str(e)}")
                # Fallback mechanism: return an empty result gracefully instead of crashing
                results.append(ProdBenchmarkResult(
                    iteration=i+1,
                    config=cfg,
                    response=f"Fallback Triggered: Error - {str(e)}",
                    success=False
                ))
                
        return results

if __name__ == "__main__":
    if not os.environ.get("OPENAI_API_KEY"):
        logger.warning("OPENAI_API_KEY not found. Ensure this is configured securely via env vars.")
    else:
        benchmarker = ProductionLLMBenchmarker()
        safe_prompt = "Explain the significance of parameter tuning in AI models."
        suite_results = benchmarker.run_benchmark_suite(prompt=safe_prompt, iterations=5)
        
        print("\n--- Advanced Production Results ---")
        for res in suite_results:
            print(f"Iter {res.iteration:02d} | Success: {res.success} | Temp: {res.config.temperature} | Resp: {res.response[:60]}...")


2026-08-20 14:26:42,256 - INFO - Running Iteration 1 with temperature=0.0 top_p=1.0 frequency_penalty=0.0


/app/.venv/lib/python3.12/site-packages/IPython/core/interactiveshell.py:3775: UserWarning: Parameters {'frequency_penalty', 'top_p'} should be specified explicitly. Instead they were passed in as part of `model_kwargs` parameter.
  exec(code_obj, self.user_global_ns, self.user_ns)


2026-08-20 14:26:42,454 - INFO - HTTP Request: POST https://api.openai.com/v1/chat/completions "HTTP/1.1 401 Unauthorized"


2026-08-20 14:26:42,456 - ERROR - Iteration 1 failed: Error code: 401 - {'error': {'message': 'Incorrect API key provided: sk-dummy. You can find your API key at https://platform.openai.com/account/api-keys.', 'type': 'invalid_request_error', 'code': 'invalid_api_key', 'param': None}, 'status': 401}


2026-08-20 14:26:42,458 - INFO - Running Iteration 2 with temperature=0.5 top_p=1.0 frequency_penalty=0.0


...

2026-08-20 14:26:42,691 - INFO - Running Iteration 5 with temperature=2.0 top_p=1.0 frequency_penalty=0.0


/app/.venv/lib/python3.12/site-packages/IPython/core/interactiveshell.py:3775: UserWarning: Parameters {'frequency_penalty', 'top_p'} should be specified explicitly. Instead they were passed in as part of `model_kwargs` parameter.
  exec(code_obj, self.user_global_ns, self.user_ns)


2026-08-20 14:26:42,775 - INFO - HTTP Request: POST https://api.openai.com/v1/chat/completions "HTTP/1.1 401 Unauthorized"


2026-08-20 14:26:42,777 - ERROR - Iteration 5 failed: Error code: 401 - {'error': {'message': 'Incorrect API key provided: sk-dummy. You can find your API key at https://platform.openai.com/account/api-keys.', 'type': 'invalid_request_error', 'code': 'invalid_api_key', 'param': None}, 'status': 401}



--- Advanced Production Results ---
Iter 01 | Success: False | Temp: 0.0 | Resp: Fallback Triggered: Error - Error code: 401 - {'error': {'me...
Iter 02 | Success: False | Temp: 0.5 | Resp: Fallback Triggered: Error - Error code: 401 - {'error': {'me...
Iter 03 | Success: False | Temp: 1.0 | Resp: Fallback Triggered: Error - Error code: 401 - {'error': {'me...
Iter 04 | Success: False | Temp: 1.5 | Resp: Fallback Triggered: Error - Error code: 401 - {'error': {'me...
Iter 05 | Success: False | Temp: 2.0 | Resp: Fallback Triggered: Error - Error code: 401 - {'error': {'me...


## Common Pitfalls in Production

1. **Altering Temperature and Top-P Simultaneously:** It is a well-known best practice to alter *either* temperature or top-p, but generally not both at the same time. If you do both, it becomes impossible to isolate which parameter caused a specific change in the output distribution.
2. **High Temperature Hallucinations:** Setting temperature too high (e.g., `> 1.5`) without constraining the output format will rapidly degrade text into absolute gibberish or severe hallucinations, as the model starts favoring highly improbable tokens.
3. **Over-Penalizing Frequency:** Setting `frequency_penalty` too high (e.g., near `2.0`) can force the model to avoid common and grammatically necessary words (like "the", "and", "is"), resulting in jarring, unreadable prose.

## Practical Lab / Homework

**Task:** Build an actionable `ParameterMatrixRunner` that takes a list of configurations, queries the LLM, and calculates a basic "Lexical Diversity Score" (Unique Words / Total Words). 

This is fully implemented below. Review the `RunMetrics` Pydantic model and the extraction of diversity metrics as a proxy for evaluating parameter variance.

In [4]:
import os
import re
from typing import List
from pydantic import BaseModel, Field
from langchain_openai import ChatOpenAI
from langchain_core.messages import HumanMessage

class LLMConfig(BaseModel):
    temperature: float = Field(default=0.7, ge=0.0, le=2.0)
    top_p: float = Field(default=1.0, ge=0.0, le=1.0)
    frequency_penalty: float = Field(default=0.0, ge=-2.0, le=2.0)

class RunMetrics(BaseModel):
    config: LLMConfig
    response_text: str
    word_count: int
    unique_word_count: int
    lexical_diversity_score: float

class ParameterMatrixRunner:
    """
    Executes a prompt across a matrix of LLM configurations and calculates diversity metrics.
    """
    def __init__(self, model_name: str = "gpt-3.5-turbo"):
        self.model_name = model_name

    def calculate_metrics(self, config: LLMConfig, text: str) -> RunMetrics:
        # Simple tokenization by word
        words = re.findall(r'\b\w+\b', text.lower())
        word_count = len(words)
        unique_words = len(set(words))
        diversity_score = unique_words / word_count if word_count > 0 else 0.0
        
        return RunMetrics(
            config=config,
            response_text=text,
            word_count=word_count,
            unique_word_count=unique_words,
            lexical_diversity_score=diversity_score
        )

    def run_matrix(self, prompt: str, configs: List[LLMConfig]) -> List[RunMetrics]:
        results: List[RunMetrics] = []
        
        for idx, config in enumerate(configs):
            print(f"Running config {idx + 1}/{len(configs)}: {config}")
            try:
                llm = ChatOpenAI(
                    model=self.model_name,
                    temperature=config.temperature,
                    model_kwargs={
                        "top_p": config.top_p,
                        "frequency_penalty": config.frequency_penalty
                    }
                )
                
                messages = [HumanMessage(content=prompt)]
                response = llm.invoke(messages)
                response_text = str(response.content)
                
                metrics = self.calculate_metrics(config, response_text)
                results.append(metrics)
                
            except Exception as e:
                print(f"Error with config {config}: {e}")
                
        return results

# --- Lab Execution ---
if __name__ == "__main__":
    if not os.environ.get("OPENAI_API_KEY"):
        print("OPENAI_API_KEY not found. Skipping Lab Execution.")
    else:
        lab_prompt = "Explain the significance of the Turing Test in modern AI in one paragraph."
        
        test_configs = [
            LLMConfig(temperature=0.0, top_p=1.0, frequency_penalty=0.0), # Baseline Greedy
            LLMConfig(temperature=1.0, top_p=1.0, frequency_penalty=0.0), # Standard
            LLMConfig(temperature=1.5, top_p=0.5, frequency_penalty=0.0), # High Temp, Constrained P
            LLMConfig(temperature=1.0, top_p=1.0, frequency_penalty=1.5), # High Penalty
        ]
        
        runner = ParameterMatrixRunner()
        lab_results = runner.run_matrix(lab_prompt, test_configs)
        
        print("\n=== Lab Results ===")
        for res in lab_results:
            print(f"\nConfig: T={res.config.temperature}, P={res.config.top_p}, FreqPen={res.config.frequency_penalty}")
            print(f"Diversity Score: {res.lexical_diversity_score:.2f} ({res.unique_word_count}/{res.word_count} unique words)")
            print(f"Response Preview: {res.response_text[:100]}...")


/app/.venv/lib/python3.12/site-packages/IPython/core/interactiveshell.py:3775: UserWarning: Parameters {'frequency_penalty', 'top_p'} should be specified explicitly. Instead they were passed in as part of `model_kwargs` parameter.
  exec(code_obj, self.user_global_ns, self.user_ns)


2026-08-20 14:26:42,875 - INFO - HTTP Request: POST https://api.openai.com/v1/chat/completions "HTTP/1.1 401 Unauthorized"


/app/.venv/lib/python3.12/site-packages/IPython/core/interactiveshell.py:3775: UserWarning: Parameters {'frequency_penalty', 'top_p'} should be specified explicitly. Instead they were passed in as part of `model_kwargs` parameter.
  exec(code_obj, self.user_global_ns, self.user_ns)


2026-08-20 14:26:42,943 - INFO - HTTP Request: POST https://api.openai.com/v1/chat/completions "HTTP/1.1 401 Unauthorized"


/app/.venv/lib/python3.12/site-packages/IPython/core/interactiveshell.py:3775: UserWarning: Parameters {'frequency_penalty', 'top_p'} should be specified explicitly. Instead they were passed in as part of `model_kwargs` parameter.
  exec(code_obj, self.user_global_ns, self.user_ns)


Running config 1/4: temperature=0.0 top_p=1.0 frequency_penalty=0.0
Error with config temperature=0.0 top_p=1.0 frequency_penalty=0.0: Error code: 401 - {'error': {'message': 'Incorrect API key provided: sk-dummy. You can find your API key at https://platform.openai.com/account/api-keys.', 'type': 'invalid_request_error', 'code': 'invalid_api_key', 'param': None}, 'status': 401}
Running config 2/4: temperature=1.0 top_p=1.0 frequency_penalty=0.0
Error with config temperature=1.0 top_p=1.0 frequency_penalty=0.0: Error code: 401 - {'error': {'message': 'Incorrect API key provided: sk-dummy. You can find your API key at https://platform.openai.com/account/api-keys.', 'type': 'invalid_request_error', 'code': 'invalid_api_key', 'param': None}, 'status': 401}
Running config 3/4: temperature=1.5 top_p=0.5 frequency_penalty=0.0


2026-08-20 14:26:43,011 - INFO - HTTP Request: POST https://api.openai.com/v1/chat/completions "HTTP/1.1 401 Unauthorized"


/app/.venv/lib/python3.12/site-packages/IPython/core/interactiveshell.py:3775: UserWarning: Parameters {'frequency_penalty', 'top_p'} should be specified explicitly. Instead they were passed in as part of `model_kwargs` parameter.
  exec(code_obj, self.user_global_ns, self.user_ns)


2026-08-20 14:26:43,084 - INFO - HTTP Request: POST https://api.openai.com/v1/chat/completions "HTTP/1.1 401 Unauthorized"


Error with config temperature=1.5 top_p=0.5 frequency_penalty=0.0: Error code: 401 - {'error': {'message': 'Incorrect API key provided: sk-dummy. You can find your API key at https://platform.openai.com/account/api-keys.', 'type': 'invalid_request_error', 'code': 'invalid_api_key', 'param': None}, 'status': 401}
Running config 4/4: temperature=1.0 top_p=1.0 frequency_penalty=1.5
Error with config temperature=1.0 top_p=1.0 frequency_penalty=1.5: Error code: 401 - {'error': {'message': 'Incorrect API key provided: sk-dummy. You can find your API key at https://platform.openai.com/account/api-keys.', 'type': 'invalid_request_error', 'code': 'invalid_api_key', 'param': None}, 'status': 401}

=== Lab Results ===


## Reference Links
- [OpenAI API Reference: Chat Completions (Temperature & Top-P)](https://platform.openai.com/docs/api-reference/chat/create)
- [LangChain OpenAI Integration Docs](https://python.langchain.com/docs/integrations/chat/openai)
- [Understanding Top-P and Temperature (HuggingFace)](https://huggingface.co/blog/how-to-generate)